# Hub Urgency — Interactive End-to-End Run

1. Run the **Setup** cell, then the **form** cell.
2. In the form, pick the **input folder** and the **output folder**, then press **Run**.
3. The report is shown under the form; the map and everything else are written to the output folder.

**Building the status overrides:** the run first writes `project_hub_links.csv` (each project UID with the hub `group`, `HubNameHE` and the H3 cells it touches) and `status_overrides_template.csv` (one row per project, `Proj_status` to edit). Tick **Link only** to stop right there, edit the template, save it as `status_overrides.csv` in the input folder and run again.

The form browses the machine the notebook runs on (on Colab: the Colab / mounted Google Drive file system, so mount Drive first).
The editable-path version of this notebook is `Hub_Urgency_End_to_End.ipynb`.

### What the input folder should contain

| File | Required | Notes |
|------|----------|-------|
| Hubs CSV | yes | Auto-detected: the CSV with `group` and `h3_index` columns |
| Inventar | yes | One `*.geojson` file **or** `geom_point.shp` / `geom_line.shp` / `geom_multiline.shp` (in the folder or one sub-folder) |
| `status_weights.csv` | yes | Columns `Proj_status`, `weight`; labels must match the inventar's `Proj_status` exactly. If missing, a pre-filled `status_weights_template.csv` is written to the output folder |
| `status_overrides.csv` | no | Columns `uid`, `Proj_status`. If the file exists those UIDs' statuses are replaced; if not, the inventar statuses are used as they are |
| `data.csv` | no | Project attributes; only needed when the inventar doesn't carry them (typical for shapefiles) |

**Report for a list of hubs:** put a CSV with a `group` column of hub ids in the input folder as `hub_selection.csv` (or set the file name under *Advanced*) and the run also writes `selected_hubs_report.xlsx`: a **Summary** sheet (run date, urgent count, one row per hub with the `hub_progress` columns and `Urgency` = `דחוף` when `status_progress_pct` ≥ the threshold, else `לא דחוף`) and a **Projects** sheet.

### What the output folder gets

Linker CSVs, `project_hub_links.csv`, `status_overrides_template.csv`, `selected_hubs_report.xlsx` (if a hub list was given), `hub_project_joined.csv`, `hub_progress.csv`, `hub_status_breakdown.csv`, `status_zero_projects.csv`, `status_weights_template.csv`, plus **`hub_urgency_report.html`** (key figures, warnings, histogram, top/bottom hubs) and **`hub_urgency_map.html`** (interactive map: hub cells by progress, projects by status). The map loads Leaflet and the basemap from the internet, so it needs a connection when opened.

## 1. Setup

In [ ]:
import importlib.util, sys

missing = [pkg for pkg, mod in [('folium', 'folium'), ('ipywidgets', 'ipywidgets'),
                                ('ipyfilechooser', 'ipyfilechooser')]
           if importlib.util.find_spec(mod) is None]
if importlib.util.find_spec('huburgency') is None:   # e.g. a fresh Colab runtime
    missing.append('git+https://github.com/oh-da/HubUrgency.git')
if missing:
    %pip install -q {' '.join(missing)}

try:  # Colab needs this for ipywidgets 8 / folder pickers
    from google.colab import output
    output.enable_custom_widget_manager()
    # from google.colab import drive; drive.mount('/content/drive')   # uncomment to browse Google Drive
except ImportError:
    pass

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from huburgency.widgets import launch_ui
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

## 2. Pick folders and run

In [ ]:
ui = launch_ui()

## 3. Status checks and plots
Run these cells **after** pressing Run in the form above.

In [ ]:
result = ui.result
assert result is not None, "Press Run in the form above first (or fix the error shown there)."
assert result.progress_df is not None, "Link-only run: nothing was scored. Untick 'Link only' and run again."

projects = result.project_df
print(f"Projects in the inventar : {len(projects):,}")
print(f"Statuses overridden      : {result.n_overridden:,}")
if result.n_overridden:
    display(projects.loc[projects['status_overridden'],
                         ['uid', 'proj_name', 'Proj_status_original', 'Proj_status']])
if result.unmapped_statuses:
    print(f"\n⚠ Statuses with NO weight (counted as 0): {result.unmapped_statuses}")
display(projects['Proj_status'].value_counts().rename('projects').to_frame())

In [ ]:
progress = result.progress_df
display(progress.sort_values('status_progress_pct', ascending=False).head(15))

with_projects = progress[progress['total_projects'] > 0]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(with_projects['status_progress_pct'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set(xlabel='Status progress (%)', ylabel='Hubs', title='Hub status progress (hubs with projects)')
axes[1].scatter(with_projects['total_projects'], with_projects['status_progress_pct'], alpha=0.6)
axes[1].set(xlabel='Projects linked to hub', ylabel='Status progress (%)', title='Progress vs. project count')
plt.tight_layout(); plt.show()

## 4. Report for another list of hubs (optional, no re-run)
Point `HUB_LIST_CSV` at a CSV with a `group` column and run the cell. It reads the results already in the output folder and writes `selected_hubs_report.xlsx` there.

In [ ]:
HUB_LIST_CSV = None   # e.g. r'/content/drive/MyDrive/Hubs_Urgency/my_hubs.csv'

if HUB_LIST_CSV:
    from huburgency.selected_hubs import selected_hubs_report
    rep = selected_hubs_report(ui.output_dir, HUB_LIST_CSV, urgent_threshold=ui._threshold.value)
    print(rep.path)
    display(rep.summary_df)